In [14]:
#Save
delayed.to_csv(
    "../data/scenario_test_results.csv",
    index=False
)

In [13]:
#View Result 
delayed[
    [
        "shipment_id",
        "supplier_id",
        "Predicted_Delay",
        "Recommendation"
    ]
].head(20)

,shipment_id,supplier_id,Predicted_Delay,Recommendation
0,1,6,1,Reduce Shipment Cost
2,3,9,1,Change Supplier
3,4,5,1,Change Supplier
4,5,18,1,Increase Inventory
5,6,12,1,Change Supplier
6,7,20,1,Increase Inventory
8,9,13,1,Change Supplier
9,10,16,1,Change Supplier
10,11,15,1,Change Supplier
11,12,5,1,Change Supplier


In [12]:
delayed["Recommendation"] = delayed.apply(recommendation, axis=1)

In [10]:
#Recommendation Column
def recommendation(row):
    if row["lead_time"] > 10:
        return "Change Supplier"

    elif row["stock_quantity"] < row["shipment_quantity"]:
        return "Increase Inventory"

    elif row["shipment_value"] > 10000:
        return "Reduce Shipment Cost"

    else:
        return "Proceed Normally"



In [9]:
#Scenario 6 – Large Customer Order

#Increase shipment quantity:
delayed["shipment_quantity"] *= 2

#Check
delayed["shipment_quantity"].sum()

np.int64(1804352)

In [8]:
#Scenario 5 – Transportation Cost Increase

#Increase shipment value by 20%:
delayed["shipment_value"] *= 1.20
#Calculate
new_cost = delayed["shipment_value"].sum()

print(new_cost)

2812069195.536


In [7]:
#Scenario 4 – Inventory Shortage
#Reduce inventory:
delayed["stock_quantity"] = 100
#Find Shortage
shortage = delayed[
    delayed["stock_quantity"] <
    delayed["shipment_quantity"]
]

shortage.head()

,shipment_id,supplier_id,product_id,shipment_date,expected_arrival,actual_arrival,shipment_quantity,shipment_status,product_name,category,...,shipment_year,shipment_quarter,shipment_weekday,is_delayed,delay_category,high_severity,large_shipment,inventory_status,shipment_value,Predicted_Delay
2,3,9,119,2026-03-17,2026-03-23,2026-03-24,325,In Transit,Maxime Product,Furniture,...,2026,1,Tuesday,1,Minor Delay,1,1,High,1558355.50,1
3,4,5,140,2025-04-20,2025-04-23,2025-04-28,419,Cancelled,Recusandae Product,Groceries,...,2025,2,Sunday,1,Major Delay,0,1,High,1133625.45,1
4,5,18,56,2025-11-20,2025-11-23,2025-11-28,136,Delayed,Error Product,Electronics,...,2025,4,Thursday,1,Major Delay,0,0,High,45312.48,1
5,6,12,48,2026-07-11,2026-07-15,2026-07-16,460,Delivered,Neque Product,Medical,...,2026,3,Saturday,1,Minor Delay,0,1,High,238408.80,1
6,7,20,142,2025-01-17,2025-01-27,2025-01-29,255,Cancelled,Dolorum Product,Groceries,...,2025,1,Friday,1,Minor Delay,0,0,High,334213.20,1


In [6]:
#Scenario 3 – Supplier Delay
#Simulate:
delayed["lead_time"] = delayed["lead_time"] + 5
#Find the worst supplier
delayed.sort_values("lead_time", ascending=False).head()

,shipment_id,supplier_id,product_id,shipment_date,expected_arrival,actual_arrival,shipment_quantity,shipment_status,product_name,category,...,shipment_year,shipment_quarter,shipment_weekday,is_delayed,delay_category,high_severity,large_shipment,inventory_status,shipment_value,Predicted_Delay
3819,3790,9,104,2025-08-13,2025-08-19,2025-08-23,376,Cancelled,Laboriosam Product,Clothing,...,2025,3,Wednesday,1,Major Delay,0,1,High,812295.36,1
3320,3296,9,109,2026-06-03,2026-06-13,2026-06-12,93,Cancelled,Beatae Product,Automotive,...,2026,2,Wednesday,0,On Time,0,0,High,108540.30,1
3359,3335,9,39,2026-01-04,2026-01-10,2026-01-08,241,Cancelled,Amet Product,Automotive,...,2026,1,Sunday,0,On Time,0,0,High,296034.76,1
3355,3331,9,100,2025-12-20,2025-12-30,2026-01-02,256,Delayed,Fugit Product,Clothing,...,2025,4,Saturday,1,Minor Delay,0,1,Medium,119203.84,1
50,51,9,103,2024-11-20,2024-11-27,2024-12-02,243,Delayed,Sunt Product,Furniture,...,2024,4,Wednesday,1,Major Delay,0,0,High,809418.42,1


In [5]:
#Scenario 2 – Capacity Constraint
#Assume warehouse capacity is
capacity = 1000
#Shipment Quantity
total_quantity = delayed["shipment_quantity"].sum()

print(total_quantity)
#Check
if total_quantity > capacity:
    print("Capacity exceeded")
else:
    print("Capacity available")

902176
Capacity exceeded


In [ ]:
#Scenario 1- Budget Reduced
#Suppose Company budget is only ₹50000
#Create
budget = 50000
#Calcualte
total_cost = delayed["shipment_value"].sum()

print(total_cost)
#Check
if total_cost > budget:
    print("Budget exceeded")
else:
    print("Budget OK")

2343390996.2799997
Budget exceeded


In [3]:
#import
import pandas as pd
import joblib
#Load Dataset
df = pd.read_csv("../data/feature_engineered_supply_chain.csv")
#Check
df.head()
df.shape
#Load Trained Model
model = joblib.load("../models/xgboost_model.pkl")
#Predict Delays 
#Feature Matrix
X = df[
    [
        "shipment_quantity",
        "unit_price",
        "lead_time",
        "stock_quantity",
        "rating",
        "shipment_value",
        "supplier_avg_delay"
    ]
]

df["Predicted_Delay"] = model.predict(X)
#Prdict
df["Predicted_Delay"] = model.predict(X)
#check
df.head()
#Only Delayed Shipment
delayed = df[df["Predicted_Delay"] == 1]
#check
delayed.head()


,shipment_id,supplier_id,product_id,shipment_date,expected_arrival,actual_arrival,shipment_quantity,shipment_status,product_name,category,...,shipment_year,shipment_quarter,shipment_weekday,is_delayed,delay_category,high_severity,large_shipment,inventory_status,shipment_value,Predicted_Delay
0,1,6,29,2025-06-03,2025-06-07,2025-06-12,16,In Transit,Architecto Product,Furniture,...,2025,2,Tuesday,1,Major Delay,0,0,High,66932.96,1
2,3,9,119,2026-03-17,2026-03-23,2026-03-24,325,In Transit,Maxime Product,Furniture,...,2026,1,Tuesday,1,Minor Delay,1,1,High,1558355.50,1
3,4,5,140,2025-04-20,2025-04-23,2025-04-28,419,Cancelled,Recusandae Product,Groceries,...,2025,2,Sunday,1,Major Delay,0,1,High,1133625.45,1
4,5,18,56,2025-11-20,2025-11-23,2025-11-28,136,Delayed,Error Product,Electronics,...,2025,4,Thursday,1,Major Delay,0,0,High,45312.48,1
5,6,12,48,2026-07-11,2026-07-15,2026-07-16,460,Delivered,Neque Product,Medical,...,2026,3,Saturday,1,Minor Delay,0,1,High,238408.80,1
